In [36]:
!pip install openai

!apt-get update
!apt-get install -y iverilog

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,632 B in 1s (3,383 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).

In [48]:
verilog_generation_prompt = '''
Return ONLY synthesizable Verilog code from "module" to "endmodule".
No markdown, no explanation.

Implement EXACTLY the following RTL structure (do not deviate):

module shift_register_ext(
  input clk,
  input reset_n,
  input data_in,
  input shift_enable,
  input shift_dir,
  output reg [7:0] data_out
);

reg [7:0] next_data_out;

always @* begin
  next_data_out = data_out;
  if (shift_enable) begin
    if (shift_dir == 1'b0)
      next_data_out = {data_in, data_out[7:1]};
    else
      next_data_out = {data_out[6:0], data_in};
  end
end

always @(posedge clk or negedge reset_n) begin
  if (!reset_n)
    data_out <= 8'b00000000;
  else
    data_out <= next_data_out;
end

endmodule
'''

In [49]:
from openai import OpenAI

client = OpenAI(
  api_key = ""
)

completion = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[{"role":"user","content":verilog_generation_prompt}],
  max_tokens=1024, # limits the maximum number of tokens (words or pieces of words) that the model will generate in its response.
  stream=False
)

print(completion.choices[0].message.content)

module shift_register_ext(
  input clk,
  input reset_n,
  input data_in,
  input shift_enable,
  input shift_dir,
  output reg [7:0] data_out
);

reg [7:0] next_data_out;

always @* begin
  next_data_out = data_out;
  if (shift_enable) begin
    if (shift_dir == 1'b0)
      next_data_out = {data_in, data_out[7:1]};
    else
      next_data_out = {data_out[6:0], data_in};
  end
end

always @(posedge clk or negedge reset_n) begin
  if (!reset_n)
    data_out <= 8'b00000000;
  else
    data_out <= next_data_out;
end

endmodule


In [50]:
completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": verilog_generation_prompt}],
    max_tokens=1024,
)

raw_llm_output = completion.choices[0].message.content
print("=== RAW LLM OUTPUT ===")
print(raw_llm_output)

=== RAW LLM OUTPUT ===
module shift_register_ext(
  input clk,
  input reset_n,
  input data_in,
  input shift_enable,
  input shift_dir,
  output reg [7:0] data_out
);

reg [7:0] next_data_out;

always @* begin
  next_data_out = data_out;
  if (shift_enable) begin
    if (shift_dir == 1'b0)
      next_data_out = {data_in, data_out[7:1]};
    else
      next_data_out = {data_out[6:0], data_in};
  end
end

always @(posedge clk or negedge reset_n) begin
  if (!reset_n)
    data_out <= 8'b00000000;
  else
    data_out <= next_data_out;
end

endmodule


In [51]:
import re

m = re.search(r'(?s)\bmodule\b.*?\bendmodule\b', raw_llm_output)
if not m:
    raise ValueError("No Verilog module found")

verilog_code = m.group(0)


In [52]:
import os

os.makedirs("shift_register_ext", exist_ok=True)
design_path = "shift_register_ext/shift_register_ext.v"

with open(design_path, "w") as f:
    f.write(verilog_code)

print("Wrote:", design_path)

Wrote: shift_register_ext/shift_register_ext.v


In [55]:
testbench_code = r'''
`timescale 1ns/1ps

module shift_register_ext_tb;

  // DUT inputs
  reg        clk;
  reg        reset_n;
  reg        data_in;
  reg        shift_enable;
  reg        shift_dir;     // 0=right, 1=left

  // DUT output
  wire [7:0] data_out;

  // Instantiate DUT
  shift_register_ext dut (
    .clk(clk),
    .reset_n(reset_n),
    .data_in(data_in),
    .shift_enable(shift_enable),
    .shift_dir(shift_dir),
    .data_out(data_out)
  );

  // Reference model
  reg [7:0] expected;

  // Clock generation: 10ns period
  initial clk = 0;
  always #5 clk = ~clk;

  // Check helper
  task check;
    input [1023:0] label;
    begin
      if (data_out !== expected) begin
        $display("FAIL [%0t] %s | expected=%b got=%b",
                 $time, label, expected, data_out);
        $finish;
      end else begin
        $display("PASS [%0t] %s | out=%b",
                 $time, label, data_out);
      end
    end
  endtask

  // Step helper: drive inputs, wait posedge, update expected, then check DUT
  task step_and_update_expected;
    input        en;
    input        dir;
    input        din;
    input [1023:0] label;
    begin
      // Drive inputs before posedge
      shift_enable = en;
      shift_dir    = dir;
      data_in      = din;

      $display("DBG pre-posedge: reset_n=%b en=%b dir=%b din=%b data_out=%b",
         reset_n, shift_enable, shift_dir, data_in, data_out);

      @(posedge clk);
      #1;
      // Update reference model
      if (!reset_n) begin
        expected = 8'b00000000;
      end else if (shift_enable) begin
        if (shift_dir == 1'b0)
          expected = {data_in, expected[7:1]};   // shift right, insert into MSB
        else
          expected = {expected[6:0], data_in};   // shift left, insert into LSB
      end
      // else hold

      check(label);
    end
  endtask

  initial begin
    // Init
    reset_n      = 1'b1;
    data_in      = 1'b0;
    shift_enable = 1'b0;
    shift_dir    = 1'b0;
    expected     = 8'b00000000;

    // -----------------------
    // 1) Initial async reset
    // -----------------------
    #1;
    reset_n = 1'b0;
    #1;
    expected = 8'b00000000;
    check("async reset asserted");

    // Release reset
    @(negedge clk);
    reset_n = 1'b1;

    // One settle cycle after reset release (IMPORTANT)
    step_and_update_expected(0, 0, 0, "post-reset settle (no shift)");

    // -----------------------
    // 2) Hold behavior
    // -----------------------
    step_and_update_expected(0, 0, 1, "hold while data_in=1");
    step_and_update_expected(0, 1, 0, "hold while dir changes");

    // -----------------------
    // 3) Shift right tests (dir=0)
    // -----------------------
    step_and_update_expected(1, 0, 1, "shift right din=1");
    step_and_update_expected(1, 0, 0, "shift right din=0");
    step_and_update_expected(1, 0, 1, "shift right din=1");
    step_and_update_expected(1, 0, 1, "shift right din=1");

    // -----------------------
    // 4) Shift left tests (dir=1)
    // -----------------------
    step_and_update_expected(1, 1, 0, "shift left din=0");
    step_and_update_expected(1, 1, 1, "shift left din=1");
    step_and_update_expected(1, 1, 1, "shift left din=1");

    // -----------------------
    // 5) Async reset mid-stream
    // -----------------------
    #1;
    reset_n = 1'b0;
    #1;
    expected = 8'b00000000;
    check("async reset mid-stream");

    // Hold reset across at least one clock edge for stability
    $display("DBG before posedge: en=%b dir=%b din=%b data_out=%b",
         shift_enable, shift_dir, data_in, data_out);

    @(posedge clk);
    expected = 8'b00000000;
    check("reset held through clock");

    // Deassert reset on negedge to avoid edge ambiguity
    @(negedge clk);
    reset_n = 1'b1;

    // *** KEY FIX: settle cycle after reset release ***
    step_and_update_expected(0, 0, 0, "post-reset settle after mid-stream reset");

    // Now start shifting again
    step_and_update_expected(1, 0, 1, "after reset shift right");

    $display("All test cases passed!");
    $finish;
  end

endmodule
'''
import os
os.makedirs("shift_register_ext", exist_ok=True)
with open("shift_register_ext/shift_register_ext_tb.v", "w") as f:
    f.write(testbench_code)

print("Wrote shift_register_ext/shift_register_ext_tb.v")


Wrote shift_register_ext/shift_register_ext_tb.v


In [56]:
with open("shift_register_ext/shift_register_ext.v", "w") as f:
    f.write(verilog_code)

print("Wrote shift_register_ext/shift_register_ext.v")


Wrote shift_register_ext/shift_register_ext.v


In [57]:
!cd shift_register_ext && iverilog -g2012 -o shift_register_ext.vvp shift_register_ext.v shift_register_ext_tb.v && vvp shift_register_ext.vvp

PASS [2000]                                                                                                             async reset asserted | out=00000000
DBG pre-posedge: reset_n=1 en=0 dir=0 din=0 data_out=00000000
PASS [16000]                                                                                                     post-reset settle (no shift) | out=00000000
DBG pre-posedge: reset_n=1 en=0 dir=0 din=1 data_out=00000000
PASS [26000]                                                                                                             hold while data_in=1 | out=00000000
DBG pre-posedge: reset_n=1 en=0 dir=1 din=0 data_out=00000000
PASS [36000]                                                                                                           hold while dir changes | out=00000000
DBG pre-posedge: reset_n=1 en=1 dir=0 din=1 data_out=00000000
PASS [46000]                                                                                                                s

In [45]:
!cd shift_register_ext && grep -n "initial" shift_register_ext.v
!cd shift_register_ext && grep -n "negedge clk" shift_register_ext.v
!cd shift_register_ext && grep -n "data_out <=" shift_register_ext.v


21:    if (!reset_n) data_out <= 8'b00000000;
22:    else          data_out <= next_data_out;
